# RAFT-Large (C_T_SKHT_V2) — DIMER dense optical flow and bounded fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/raft-optical-flow-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/raft-optical-flow-pipeline/blob/main/tutorials/raft_optical_flow_colab.ipynb) [![torchvision](https://img.shields.io/badge/torchvision-raft__large-ee4c2c?style=flat&logo=pytorch&logoColor=white)](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.optical_flow.raft_large.html) [![Upstream](https://img.shields.io/badge/Upstream-pytorch%2Fvision-181717?style=flat&logo=github&logoColor=white)](https://github.com/pytorch/vision) [![arXiv](https://img.shields.io/badge/arXiv-2003.12039-b31b1b.svg)](https://arxiv.org/abs/2003.12039) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/kurtvalcorza/raft-optical-flow-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.1 — **standalone** (§4)  
**Capability:** dense optical flow between two frames with torchvision's RAFT-Large, and a bounded fine-tune on frame pairs with known flow that is scored on a held-out split by end-point error against a zero-flow baseline and exported as a reloadable SafeTensors adapter

**This notebook is standalone.** It carries the repository's package (2 modules under `src/raft_optical_flow_pipeline/`, at revision `a14726e1875d`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and `download.pytorch.org` for the checkpoint file pinned by SHA-256 `unpinned` (one file that torchvision's weight metadata lists as 20.129 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned checkpoint, estimates flow on a rendered pair with exact ground truth, validates the 40-pair dataset, splits it into training and held-out parts, measures the pre-adaptation baseline, **runs the bounded fine-tune**, re-evaluates on the held-out split, estimates flow on unseen pairs, exports the adapter, reloads it onto a fresh base model to verify the flow, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone or DIMER worker is required (NOTEBOOK_SPEC 2.1 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included, and both are off by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own two frames through the same validation, estimation and evaluation-report stages as the sample pair. `USE_BYOD_DATASET` takes your own frame pairs with ground-truth flow through the full adaptation workflow — validate, split, baseline, fine-tune, evaluate, export and reload — under NOTEBOOK_SPEC 2.1 DAT14. Set `BYOD_IMAGE1_PATH` and `BYOD_IMAGE2_PATH`, or `BYOD_DATASET_DIR`, to read from a location without an upload dialog (EXE2).

Optical flow is the apparent motion of every pixel between two frames: for each pixel `(x, y)` of frame 1, a displacement `(u, v)` in pixels such that the same scene point appears at `(x + u, y + v)` in frame 2. RAFT (Recurrent All-Pairs Field Transforms) computes it in three parts. A **feature encoder** turns each frame into features at 1/8 resolution; RAFT correlates every feature of frame 1 with every feature of frame 2 into a 4-D correlation volume; and a **recurrent update block** starts from zero flow and refines it a fixed number of times, looking up the volume around the current estimate at each step. A learned upsampler returns the flow at full resolution.

This notebook uses torchvision's `raft_large` builder with its default weights, `Raft_Large_Weights.C_T_SKHT_V2`. torchvision's weight documentation says these weights were trained from scratch on FlyingChairs and FlyingThings3D, then fine-tuned on a mix of Sintel, KITTI, HD1K and FlyingThings3D (clean pass), and lists Sintel test end-point errors of 1.819 (clean pass) and 3.067 (final pass).

**The default path really adapts the model:** it builds a labelled dataset of textured frame pairs with exact flow, measures the pretrained end-point error on a held-out split next to a zero-flow baseline, runs a bounded fine-tune with RAFT's own sequence loss, re-scores the held-out split, runs the adapted model on unseen pairs, exports the changed tensors as a SafeTensors adapter, and reloads that adapter onto a fresh copy of the verified base model to check that it reproduces the same flow. Every number you see is measured in this notebook runtime.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the pinned checkpoint; estimate flow on a frame pair with exact ground truth and score it by end-point error (EPE), outlier rates and angular error against the zero-flow baseline; see how the number of recurrent updates changes the estimate; probe the model with a static and a featureless pair; build and validate a labelled flow dataset; split it and measure a pre-adaptation baseline; run a bounded fine-tune with RAFT's sequence loss; score the adapted model on the held-out split; run inference on unseen pairs; and export, reload and verify the adapter.

**This notebook does not demonstrate:** real-video optical flow claims (the adaptation dataset is rendered in code with integer displacements, so the model learns these renderings and nothing about camera footage); Sintel or KITTI benchmark results (the metrics helper follows the benchmark definitions but is run on synthetic data only); full-schedule RAFT training (torchvision's reference recipe trains for hundreds of thousands of steps on several GPUs; the tutorial runs a few epochs on 30 pairs); stereo disparity; scene flow; video interpolation.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). A CUDA GPU such as a Colab or Kaggle T4 is the documented runtime for the fine-tuning stages and is used automatically when present; the notebook also runs on CPU, much more slowly. Runtimes are not measured in this revision. The pinned `torch==2.14.0` wheel is the largest download.
- **Knowledge:** basic Python, PIL and NumPy; images as `(H, W)` pixel grids; a flow field as an `(H, W, 2)` array of `(u, v)` displacements in pixels; and the Euclidean distance between two vectors.
- **Data:** the default path generates everything in code with `samples.py` and downloads no dataset: one 256×256 demonstration pair and a 40-pair labelled dataset, each with exact flow and a validity mask. BYOD is optional and off by default. Expected BYOD input: two frames of one size, or a directory holding `pairs.json` — a list of `{'frame1': 'a.png', 'frame2': 'b.png', 'flow': 'a.flo', 'valid': 'a_valid.png'}` objects, where `flow` is a Middlebury `.flo` file or a `.npy` array of shape `(H, W, 2)` and `valid` is optional — and the files it names. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** `download.pytorch.org` only, to fetch the pinned `torchvision/raft_large` checkpoint (one file that torchvision's weight metadata lists as 20.129 MB) at revision `unpinned…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchvision`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'raft-optical-flow-pipeline',
    'repository_revision': 'a14726e1875d94e92519c11eeff27ce619d851ef',
    'embedded_module': 'src/raft_optical_flow_pipeline/pipeline.py',
    'embedded_modules': ['src/raft_optical_flow_pipeline/samples.py', 'src/raft_optical_flow_pipeline/pipeline.py'],
    'module_sha256': '766061dedb0747874e215325834b19d776d531ba2da85ea6da6023cacafb7ee4',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchvision, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/raft_optical_flow_pipeline/` @ `a14726e1875d`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/raft_optical_flow_pipeline/samples.py`

In [ ]:
"""Deterministic in-code frame pairs with exact optical flow, and Middlebury ``.flo`` input/output.

Nothing here is downloaded and nothing needs torch — numpy and Pillow only. Each pair is rendered from a
larger textured canvas:

* the **background** is a window into the canvas; frame 2's window is shifted by a whole number of pixels,
  so every background pixel moves by the same known displacement;
* each **object** is a textured disc or square pasted at one position in frame 1 and at a displaced
  position in frame 2, so its pixels move by that object's own known displacement.

The flow is defined from frame 1 to frame 2: the pixel at ``(x, y)`` in frame 1 appears at
``(x + u, y + v)`` in frame 2. A pixel is **valid** when its correspondence is visible in frame 2: object
pixels always are (objects stay inside the frame and do not overlap), background pixels are not when an
object covers their destination or when the destination leaves the frame. Scores are computed on valid
pixels only.
"""

from __future__ import annotations

from typing import Any

import numpy as np
from PIL import Image

PAIR_SIZE = (256, 256)  # (width, height), a multiple of 8 as RAFT requires
MAX_OBJECTS = 4
FLO_MAGIC = 202021.25  # Middlebury .flo header tag


def _texture(rng: np.random.Generator, height: int, width: int) -> np.ndarray:
    """Band-limited colour noise: enough structure for matching at every scale RAFT looks at."""
    base = rng.integers(0, 256, (height // 8 + 2, width // 8 + 2, 3), dtype=np.uint8)
    coarse = np.asarray(Image.fromarray(base).resize((width, height), Image.BICUBIC), dtype=np.float32)
    fine = rng.integers(0, 64, (height, width, 3)).astype(np.float32)
    return np.clip(0.8 * coarse + fine, 0, 255).astype(np.uint8)


def _shape_mask(kind: str, radius: int) -> np.ndarray:
    side = 2 * radius + 1
    yy, xx = np.mgrid[:side, :side]
    if kind == "disc":
        return (xx - radius) ** 2 + (yy - radius) ** 2 <= radius**2
    return np.ones((side, side), dtype=bool)


def moving_shapes_pair(
    seed: int = 0,
    *,
    size: tuple[int, int] = PAIR_SIZE,
    max_background_shift: int = 6,
    max_object_shift: int = 12,
    n_objects: int | None = None,
) -> dict[str, Any]:
    """One frame pair with its exact flow ``(H, W, 2)`` float32 (u, v) and boolean valid mask ``(H, W)``."""
    width, height = size
    if width % 8 or height % 8 or min(width, height) < 128:
        raise ValueError(f"size must be multiples of 8 and at least 128 px, got {size}")
    rng = np.random.default_rng(seed)
    margin = max_background_shift
    canvas = _texture(rng, height + 2 * margin, width + 2 * margin)
    bx, by = (int(v) for v in rng.integers(-max_background_shift, max_background_shift + 1, 2))
    frame1 = canvas[margin : margin + height, margin : margin + width].copy()
    frame2 = canvas[margin + by : margin + by + height, margin + bx : margin + bx + width].copy()
    # A window shifted by (bx, by) shows the content moved by (-bx, -by).
    flow = np.zeros((height, width, 2), dtype=np.float32)
    flow[..., 0], flow[..., 1] = -bx, -by
    ys, xs = np.mgrid[:height, :width]
    valid = (xs - bx >= 0) & (xs - bx < width) & (ys - by >= 0) & (ys - by < height)

    count = int(rng.integers(1, MAX_OBJECTS + 1)) if n_objects is None else n_objects
    placed: list[tuple[int, int, int]] = []
    covered2 = np.zeros((height, width), dtype=bool)
    objects = []
    attempts = 0
    while len(objects) < count and attempts < 200:
        attempts += 1
        radius = int(rng.integers(14, 34))
        dx, dy = (int(v) for v in rng.integers(-max_object_shift, max_object_shift + 1, 2))
        lo = radius + max_object_shift + 1
        cx, cy = int(rng.integers(lo, width - lo)), int(rng.integers(lo, height - lo))
        reach = radius + max_object_shift + 2
        if any(abs(cx - px) < reach + pr and abs(cy - py) < reach + pr for px, py, pr in placed):
            continue
        placed.append((cx, cy, radius))
        kind = "disc" if rng.random() < 0.5 else "square"
        mask = _shape_mask(kind, radius)
        patch = _texture(rng, mask.shape[0], mask.shape[1])
        y0, x0 = cy - radius, cx - radius
        region1 = (slice(y0, y0 + mask.shape[0]), slice(x0, x0 + mask.shape[1]))
        region2 = (slice(y0 + dy, y0 + dy + mask.shape[0]), slice(x0 + dx, x0 + dx + mask.shape[1]))
        frame1[region1][mask] = patch[mask]
        frame2[region2][mask] = patch[mask]
        flow[region1][mask] = (dx, dy)
        valid[region1] |= mask
        covered2[region2] |= mask
        objects.append({"kind": kind, "center": [cx, cy], "radius": radius, "shift": [dx, dy]})

    # A background pixel whose destination is covered by an object in frame 2 is occluded.
    object_pixels = np.zeros((height, width), dtype=bool)
    for obj in objects:
        cx, cy, r = obj["center"][0], obj["center"][1], obj["radius"]
        m = _shape_mask(obj["kind"], r)
        object_pixels[cy - r : cy + r + 1, cx - r : cx + r + 1] |= m
    dest_x = np.clip(xs - bx, 0, width - 1)
    dest_y = np.clip(ys - by, 0, height - 1)
    occluded = ~object_pixels & covered2[dest_y, dest_x]
    valid &= ~occluded
    return {
        "id": f"pair-{seed}",
        "image1": Image.fromarray(frame1),
        "image2": Image.fromarray(frame2),
        "flow": flow,
        "valid": valid,
        "background_shift": [-bx, -by],
        "objects": objects,
    }


def flow_dataset(
    n_pairs: int = 40, *, seed: int = 0, size: tuple[int, int] = PAIR_SIZE
) -> list[dict[str, Any]]:
    """A deterministic list of ``moving_shapes_pair`` records, the adaptation dataset of the tutorial."""
    if not 1 <= n_pairs <= 1000:
        raise ValueError(f"n_pairs must be in 1..1000, got {n_pairs}")
    return [moving_shapes_pair(seed * 100_000 + i, size=size) for i in range(n_pairs)]


def split_pairs(
    records: list[dict[str, Any]], *, train_fraction: float = 0.75, seed: int = 0
) -> tuple[list, list]:
    """Seeded random split at the pair level; pairs cut from the same video must be split by video instead."""
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    if len(records) < 2:
        raise ValueError(f"at least 2 records are required to split, got {len(records)}")
    order = np.random.default_rng(seed).permutation(len(records))
    n_train = min(len(records) - 1, max(1, int(len(records) * train_fraction)))
    chosen = {int(i) for i in order[:n_train]}
    return [r for i, r in enumerate(records) if i in chosen], [
        r for i, r in enumerate(records) if i not in chosen
    ]


def static_pair(size: tuple[int, int] = PAIR_SIZE, seed: int = 7) -> dict[str, Any]:
    """The same textured frame twice: the true flow is zero everywhere."""
    width, height = size
    frame = Image.fromarray(_texture(np.random.default_rng(seed), height, width))
    return {
        "id": "static",
        "image1": frame,
        "image2": frame.copy(),
        "flow": np.zeros((height, width, 2), np.float32),
        "valid": np.ones((height, width), bool),
    }


def blank_pair(size: tuple[int, int] = PAIR_SIZE) -> dict[str, Any]:
    """Two featureless white frames: any motion is undetermined, and a flow estimate is a guess."""
    width, height = size
    frame = Image.new("RGB", size, (255, 255, 255))
    return {
        "id": "blank",
        "image1": frame,
        "image2": frame.copy(),
        "flow": np.zeros((height, width, 2), np.float32),
        "valid": np.ones((height, width), bool),
    }


def read_flo(path: Any) -> np.ndarray:
    """Read a Middlebury ``.flo`` file into an ``(H, W, 2)`` float32 array; refuse a bad header or size."""
    with open(path, "rb") as fh:
        magic = np.fromfile(fh, np.float32, count=1)
        if magic.size != 1 or magic[0] != FLO_MAGIC:
            raise ValueError(f"{path}: not a Middlebury .flo file (bad magic)")
        width, height = (int(v) for v in np.fromfile(fh, np.int32, count=2))
        if not (0 < width <= 4096 and 0 < height <= 4096):
            raise ValueError(f"{path}: implausible .flo size {width}x{height}")
        data = np.fromfile(fh, np.float32, count=2 * width * height)
    if data.size != 2 * width * height:
        raise ValueError(f"{path}: truncated .flo data")
    return data.reshape(height, width, 2)


def write_flo(path: Any, flow: np.ndarray) -> None:
    """Write an ``(H, W, 2)`` flow array as a Middlebury ``.flo`` file."""
    flow = np.asarray(flow, dtype=np.float32)
    if flow.ndim != 3 or flow.shape[2] != 2:
        raise ValueError(f"flow must have shape (H, W, 2), got {flow.shape}")
    with open(path, "wb") as fh:
        np.array([FLO_MAGIC], np.float32).tofile(fh)
        np.array([flow.shape[1], flow.shape[0]], np.int32).tofile(fh)
        flow.tofile(fh)

**Module 2/2:** `src/raft_optical_flow_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Dense optical flow and bounded fine-tuning with torchvision's RAFT-Large (``C_T_SKHT_V2`` weights).

The model is torchvision's own ``raft_large`` architecture, built with ``weights=None`` so that nothing is
downloaded at construction, then loaded from one digest-verified local checkpoint file (``weights/<key>/``)
with ``torch.load(..., weights_only=True)``, which refuses arbitrary pickled objects.

Until ``tools/pin_snapshot.py`` has recorded the checkpoint's SHA-256 and byte size, the package refuses to
stage, verify or load weights: an unpinned checkpoint is never trusted.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .samples import read_flo` removed — names are kernel globals defined by the carried modules

MODEL_ID = "torchvision/raft_large"
# The pinned identity of a URL-hosted checkpoint is the SHA-256 of its bytes. torchvision names the file
# after the first 8 hex digits of that digest, so the URL is content-addressed; the full digest is pinned.
MODEL_REVISION = "unpinned"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "raft-large-c-t-skht-v2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "raft-adapter-v1"
UNPINNED = "unpinned"
PIN_COMMAND = "python tools/pin_snapshot.py"
WEIGHTS_FILE = "raft_large_C_T_SKHT_V2-ff5fadd5.pth"
WEIGHTS_URL = f"https://download.pytorch.org/models/{WEIGHTS_FILE}"
WEIGHTS_SHA256_PREFIX = "ff5fadd5"
TORCHVISION_WEIGHTS = "Raft_Large_Weights.C_T_SKHT_V2"

# Operational ceilings. RAFT's all-pairs correlation volume holds (H/8 * W/8)^2 values, so memory grows with
# the square of the pixel count: 1024 x 768 already needs about 0.6 GB for it alone.
MIN_IMAGE_SIDE = 128  # torchvision's weight metadata: min_size (128, 128)
MAX_IMAGE_SIDE = 1024
MAX_PIXELS = 1024 * 768
MAX_RECORDS = 2000
DEFAULT_FLOW_UPDATES = 12  # torchvision's default num_flow_updates
MAX_FLOW = 400.0  # the reference sequence loss ignores ground-truth displacements at or above this (pixels)

# Bounded fine-tuning defaults, following torchvision's optical-flow reference (references/optical_flow):
# AdamW, sequence loss with gamma 0.8, gradient-norm clipping at 1.0, BatchNorm frozen as in its Sintel stage.
DEFAULT_EPOCHS = 3
DEFAULT_BATCH_SIZE = 2
DEFAULT_LEARNING_RATE = 2e-5
DEFAULT_WEIGHT_DECAY = 5e-5
DEFAULT_GAMMA = 0.8
DEFAULT_TRAIN_UPDATES = 12
DEFAULT_SEED = 20260925
ENCODER_PREFIXES = ("feature_encoder.", "context_encoder.")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_pinned() -> bool:
    """True once MODEL_REVISION is the checkpoint's 64-hex SHA-256."""
    revision = MODEL_REVISION
    return len(revision) == 64 and all(c in "0123456789abcdef" for c in revision)


def _require_pinned(action: str) -> None:
    if not is_pinned():
        raise RuntimeError(
            f"refusing to {action}: {MODEL_ID} has no pinned checkpoint digest yet (MODEL_REVISION = "
            f"{MODEL_REVISION!r}); run `{PIN_COMMAND}` to record the SHA-256 and byte size"
        )


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local checkpoint against its DIMER manifest; raise naming the first mismatch."""
    _require_pinned("verify the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        if not entry.get("sha256") or entry.get("bytes") is None:
            raise ValueError(f"{entry['path']}: manifest records no sha256/bytes; run `{PIN_COMMAND}`")
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"checkpoint file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _url_download(entry: Mapping[str, Any], root: Path) -> None:
    """Fetch one manifest entry from its URL; torch.hub checks the filename's SHA-256 prefix on the way in."""
    from torch.hub import download_url_to_file

    download_url_to_file(
        entry["url"], str(root / entry["path"]), hash_prefix=WEIGHTS_SHA256_PREFIX, progress=False
    )


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[Mapping[str, Any], Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    _require_pinned("stage the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"checkpoint at {root} is missing {[e['path'] for e in missing]}; "
            "pass allow_download=True to fetch it"
        )
    fetch = downloader or _url_download
    for entry in missing:
        if entry.get("url") != WEIGHTS_URL:
            raise ValueError(f"manifest URL {entry.get('url')!r} != package URL {WEIGHTS_URL!r}")
        fetch(entry, root)
    return [entry["path"] for entry in missing]


# --------------------------------------------------------------------------- metrics


def flow_metrics(predicted: np.ndarray, truth: np.ndarray, valid: np.ndarray | None = None) -> dict[str, Any]:
    """End-point error (EPE), outlier rates and angular error of a flow field against ground truth.

    ``EPE`` is the mean Euclidean distance in pixels between predicted and true displacement over valid
    pixels. ``1px``/``3px``/``5px`` are the fractions of valid pixels whose error is below that many pixels
    (higher is better), as torchvision's reference evaluation reports them. ``angular_error_deg`` is the mean
    angle between the space-time vectors (u, v, 1), the Middlebury convention.
    """
    predicted = np.asarray(predicted, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if predicted.shape != truth.shape or truth.ndim != 3 or truth.shape[2] != 2:
        raise ValueError(f"flows must both have shape (H, W, 2), got {predicted.shape} and {truth.shape}")
    mask = np.ones(truth.shape[:2], dtype=bool) if valid is None else np.asarray(valid, dtype=bool)
    if mask.shape != truth.shape[:2]:
        raise ValueError(f"valid mask shape {mask.shape} != flow shape {truth.shape[:2]}")
    mask = mask & (np.linalg.norm(truth, axis=2) < MAX_FLOW)
    if not mask.any():
        raise ValueError("no valid pixel to score")
    error = np.linalg.norm(predicted - truth, axis=2)[mask]
    p, t = predicted[mask], truth[mask]
    # Angle between (u, v, 1) vectors as atan2(|a x b|, a . b): exact near 0, where arccos is not.
    a = np.column_stack([p, np.ones(len(p))])
    b = np.column_stack([t, np.ones(len(t))])
    angle = np.arctan2(np.linalg.norm(np.cross(a, b), axis=1), (a * b).sum(axis=1))
    return {
        "epe": float(error.mean()),
        "1px": float((error < 1).mean()),
        "3px": float((error < 3).mean()),
        "5px": float((error < 5).mean()),
        "angular_error_deg": float(np.degrees(angle).mean()),
        "valid_pixels": int(mask.sum()),
    }


def aggregate_metrics(rows: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Mean of per-pair metrics (each pair weighs the same) plus the pixel-weighted EPE."""
    if not rows:
        raise ValueError("no rows to aggregate")
    keys = ("epe", "1px", "3px", "5px", "angular_error_deg")
    out: dict[str, Any] = {key: float(np.mean([row[key] for row in rows])) for key in keys}
    pixels = sum(row["valid_pixels"] for row in rows)
    out["pixel_weighted_epe"] = float(sum(row["epe"] * row["valid_pixels"] for row in rows) / pixels)
    out["n_pairs"] = len(rows)
    return out


# --------------------------------------------------------------------------- input validation


def validate_pair(image1: Any, image2: Any) -> tuple[Image.Image, Image.Image]:
    """Two PIL images of one size inside the ceilings; returns them converted to RGB."""
    for name, image in (("image1", image1), ("image2", image2)):
        if not isinstance(image, Image.Image):
            raise TypeError(f"{name} must be a PIL.Image.Image, got {type(image).__name__}")
    if image1.size != image2.size:
        raise ValueError(f"frames differ in size: {image1.size} vs {image2.size}")
    width, height = image1.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"frame side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"frame side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    if width * height > MAX_PIXELS:
        raise ValueError(f"frame has {width * height} pixels > MAX_PIXELS {MAX_PIXELS}")
    return image1.convert("RGB"), image2.convert("RGB")


def _check_updates(value: Any) -> int:
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 32:
        raise ValueError(f"num_flow_updates must be an int in 1..32, got {value!r}")
    return value


INPUT_SCHEMA: dict[str, Any] = {
    "input": "two PIL.Image.Image frames of the same size (any mode, converted to RGB); frame 1 then frame 2",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "max_pixels": MAX_PIXELS,
    "num_flow_updates": [1, 32],
    "output": "flow (H, W, 2) float32 in pixels: frame-1 pixel (x, y) moves to (x + u, y + v) in frame 2",
    "preprocessing": (
        "RGB to float in [0, 1], normalised to [-1, 1] (torchvision's OpticalFlow preset); frames are padded "
        "by edge replication to a multiple of 8 px and the flow is cropped back"
    ),
}


def validate_inputs(
    image1: Any, image2: Any, *, num_flow_updates: int = DEFAULT_FLOW_UPDATES
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict)."""
    first, second = validate_pair(image1, image2)
    updates = _check_updates(num_flow_updates)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": name, "mode": img.mode, "size": list(img.size)}
            for name, img in (("frame1", image1), ("frame2", image2))
        ],
        "num_flow_updates": updates,
        "padding": [(-first.width) % 8, (-first.height) % 8],
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(records: Sequence[Mapping[str, Any]], *, epochs: int = DEFAULT_EPOCHS) -> dict[str, Any]:
    """Validation stage for flow records ``{"image1", "image2", "flow", "valid"?}``: raise on the first broken
    record, else return the dataset manifest with displacement statistics."""
    if not records:
        raise ValueError("dataset must hold at least one record")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset holds {len(records)} records > MAX_RECORDS {MAX_RECORDS}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be an int in 1..100, got {epochs!r}")
    magnitudes, valid_fraction, sizes = [], [], set()
    findings = []
    for idx, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"image1", "image2", "flow"} <= set(record):
            raise ValueError(f"record {idx} must be a mapping with 'image1', 'image2' and 'flow'")
        try:
            first, _second = validate_pair(record["image1"], record["image2"])
        except (TypeError, ValueError) as exc:
            raise type(exc)(f"record {idx}: {exc}") from exc
        flow = np.asarray(record["flow"])
        if flow.shape != (first.height, first.width, 2):
            raise ValueError(f"record {idx}: flow shape {flow.shape} != ({first.height}, {first.width}, 2)")
        if not np.isfinite(flow).all():
            raise ValueError(f"record {idx}: flow contains non-finite values")
        valid = np.asarray(record.get("valid", np.ones(flow.shape[:2], bool)))
        if valid.shape != flow.shape[:2] or (valid.dtype != bool and not np.isin(valid, (0, 1)).all()):
            raise ValueError(f"record {idx}: valid mask must be boolean with shape {flow.shape[:2]}")
        valid = valid.astype(bool)
        if not valid.any():
            raise ValueError(f"record {idx}: valid mask selects no pixel")
        magnitude = np.linalg.norm(flow, axis=2)[valid]
        if (magnitude >= MAX_FLOW).any():
            findings.append(
                f"record {idx}: {int((magnitude >= MAX_FLOW).sum())} valid pixels move >= {MAX_FLOW} px "
                "and are ignored by the loss"
            )
        magnitudes.append(float(magnitude.mean()))
        valid_fraction.append(float(valid.mean()))
        sizes.add(first.size)
    if len(sizes) > 1:
        findings.append(f"{len(sizes)} different frame sizes; batches are grouped by size")
    return {
        "n_records": len(records),
        "frame_sizes": sorted([list(s) for s in sizes]),
        "mean_displacement_px": float(np.mean(magnitudes)),
        "max_mean_displacement_px": float(np.max(magnitudes)),
        "mean_valid_fraction": float(np.mean(valid_fraction)),
        "epochs": epochs,
        "findings": findings,
        "verdict": "accepted",
    }


def read_flow_records(directory: str | Path) -> list[dict[str, Any]]:
    """Read BYOD flow records from ``<directory>/pairs.json`` plus the files it names.

    ``pairs.json`` is a list of objects such as
    ``{"frame1": "a.png", "frame2": "b.png", "flow": "a.flo", "valid": "a_valid.png"}``; ``flow`` is a
    Middlebury ``.flo`` file or a ``.npy`` array of shape (H, W, 2), and ``valid`` is optional (a
    single-channel PNG whose non-zero pixels are valid). File names must stay inside ``directory``.
    """
    root = Path(directory).resolve()
    index = root / "pairs.json"
    if not index.is_file():
        raise FileNotFoundError(f"{index} not found; expected pairs.json next to the frames")
    entries = json.loads(index.read_text(encoding="utf-8"))
    if not isinstance(entries, list) or not entries:
        raise ValueError("pairs.json must hold a non-empty list of records")
    if len(entries) > MAX_RECORDS:
        raise ValueError(f"pairs.json lists {len(entries)} records > MAX_RECORDS {MAX_RECORDS}")

    def resolve(idx: int, name: Any) -> Path:
        relative = Path(str(name))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"pairs.json entry {idx}: file {name!r} must be relative to {root}")
        resolved = (root / relative).resolve()
        if root not in resolved.parents:
            raise ValueError(f"pairs.json entry {idx}: file {name!r} resolves outside {root}")
        return resolved

    records = []
    for idx, entry in enumerate(entries):
        if not isinstance(entry, dict) or not {"frame1", "frame2", "flow"} <= set(entry):
            raise ValueError(f"pairs.json entry {idx} must have 'frame1', 'frame2' and 'flow'")
        paths = {key: resolve(idx, entry[key]) for key in ("frame1", "frame2", "flow")}
        frames = []
        for key in ("frame1", "frame2"):
            with Image.open(paths[key]) as handle:
                frames.append(handle.convert("RGB"))
        flow = (
            np.load(paths["flow"], allow_pickle=False)
            if paths["flow"].suffix == ".npy"
            else read_flo(paths["flow"])
        )
        record = {
            "id": str(entry["frame1"]),
            "image1": frames[0],
            "image2": frames[1],
            "flow": flow.astype(np.float32),
        }
        if entry.get("valid"):
            with Image.open(resolve(idx, entry["valid"])) as handle:
                record["valid"] = np.asarray(handle.convert("L")) > 0
        records.append(record)
    return records


def evaluation_report(
    result: Mapping[str, Any],
    truth: np.ndarray | None = None,
    valid: np.ndarray | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-pair evaluation stage: EPE, outlier rates and angular error against ground truth, next to
    the zero-flow baseline (predicting no motion), or ``not-measurable`` without ground truth."""
    flow = np.asarray(result["flow"])
    base = {
        "task": "dense optical flow between two frames",
        "sample_kind": sample_kind,
        "flow_shape": list(flow.shape),
        "num_flow_updates": result.get("num_flow_updates", DEFAULT_FLOW_UPDATES),
        "mean_predicted_displacement_px": float(np.linalg.norm(flow, axis=2).mean()),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if truth is None:
        return {
            **base,
            "metrics": {},
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth flow was supplied for the evaluated pair",
            "needs": (
                "frame pairs from your own footage with ground-truth flow (rendered or measured), "
                "scored by EPE and outlier rates against the zero-flow baseline"
            ),
        }
    metrics = flow_metrics(flow, truth, valid)
    baseline = flow_metrics(np.zeros_like(np.asarray(truth, dtype=np.float32)), truth, valid)
    return {
        **base,
        "metrics": metrics,
        "baselines": [{"id": "zero-flow", **baseline}],
        "verdict": "sample-sanity",
        "reason": "one frame pair with exact ground truth; geometry sanity evidence, not a benchmark",
        "needs": "a labelled set of frame pairs from the deployment domain for any EPE claim",
    }


# --------------------------------------------------------------------------- model


def build_model() -> Any:
    """torchvision's raft_large architecture with random weights and nothing downloaded."""
    from torchvision.models.optical_flow import raft_large

    return raft_large(weights=None, progress=False)


def _to_tensor(images: Sequence[Image.Image]) -> Any:
    """torchvision's OpticalFlow preset: uint8 RGB to float in [0, 1], then normalised to [-1, 1]."""
    import torch

    arrays = np.stack([np.asarray(image, dtype=np.float32) / 255.0 for image in images])
    return torch.from_numpy(arrays).permute(0, 3, 1, 2) * 2.0 - 1.0


def _pad8(batch: Any) -> tuple[Any, tuple[int, int]]:
    import torch.nn.functional as F

    height, width = batch.shape[-2:]
    pad_h, pad_w = (-height) % 8, (-width) % 8
    if pad_h or pad_w:
        batch = F.pad(batch, (0, pad_w, 0, pad_h), mode="replicate")
    return batch, (height, width)


def _hold_batchnorm(module: Any) -> None:
    """Keep every BatchNorm layer under ``module`` in eval mode (RAFT's context encoder carries BatchNorm)."""
    import torch

    for sub in module.modules():
        if isinstance(sub, torch.nn.modules.batchnorm._BatchNorm):
            sub.eval()


def sequence_loss(predictions: Sequence[Any], truth: Any, valid: Any, gamma: float = DEFAULT_GAMMA) -> Any:
    """RAFT's loss: L1 flow error over every refinement, weighted gamma**(N - i), valid and < MAX_FLOW px only
    (torchvision references/optical_flow utils.sequence_loss)."""
    import torch

    if not 0.0 < gamma < 1.0:
        raise ValueError(f"gamma must be in (0, 1), got {gamma}")
    norm = torch.sum(truth**2, dim=1).sqrt()
    mask = (valid & (norm < MAX_FLOW))[:, None, :, :]
    stacked = torch.stack(list(predictions))
    per_iteration = ((stacked - truth).abs() * mask).mean(dim=(1, 2, 3, 4))
    weights = gamma ** torch.arange(
        len(predictions) - 1, -1, -1, device=truth.device, dtype=per_iteration.dtype
    )
    return (per_iteration * weights).sum()


@dataclass
class RaftPipeline:
    """Dense optical flow, and bounded fine-tuning on a caller's flow pairs, over RAFT-Large."""

    model: Any
    device: str
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    frozen_prefixes: tuple[str, ...] = field(default_factory=tuple)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> RaftPipeline:
        _require_pinned("load the model")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no checkpoint manifest at {root}; stage {MODEL_ID} under weights/{MODEL_KEY} "
                "(allow_download=True fetches the manifest-listed file)"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)

        import torch

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = build_model()
        state = torch.load(root / WEIGHTS_FILE, map_location="cpu", weights_only=True)
        model.load_state_dict(state, strict=True)
        return cls(
            model=model.to(resolved_device).eval(),
            device=resolved_device,
            source=str(root),
            base_state_digest=MODEL_REVISION,
        )

    def _forward(
        self, first: Sequence[Image.Image], second: Sequence[Image.Image], updates: int
    ) -> list[Any]:
        batch1, (height, width) = _pad8(_to_tensor(first).to(self.device))
        batch2, _ = _pad8(_to_tensor(second).to(self.device))
        predictions = self.model(batch1, batch2, num_flow_updates=updates)
        return [flow[..., :height, :width] for flow in predictions]

    def estimate(
        self, image1: Image.Image, image2: Image.Image, *, num_flow_updates: int = DEFAULT_FLOW_UPDATES
    ) -> dict[str, Any]:
        """Estimate the flow from ``image1`` to ``image2``; returns ``flow`` as an (H, W, 2) float32 array."""
        import torch

        first, second = validate_pair(image1, image2)
        updates = _check_updates(num_flow_updates)
        was_training = self.model.training
        self.model.eval()
        with torch.inference_mode():
            flow = self._forward([first], [second], updates)[-1][0]
        if was_training:
            self.model.train()
        array = flow.permute(1, 2, 0).float().cpu().numpy()
        if array.shape != (first.height, first.width, 2) or not np.isfinite(array).all():
            raise RuntimeError(f"backend returned a malformed flow of shape {array.shape}")
        return {
            "flow": array,
            "num_flow_updates": updates,
            "width": first.width,
            "height": first.height,
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, num_flow_updates: int = DEFAULT_FLOW_UPDATES
    ) -> dict[str, Any]:
        """Mean EPE, outlier rates and angular error over labelled pairs, next to the zero-flow baseline."""
        rows, zero_rows = [], []
        for record in records:
            flow = self.estimate(record["image1"], record["image2"], num_flow_updates=num_flow_updates)[
                "flow"
            ]
            truth = np.asarray(record["flow"], dtype=np.float32)
            valid = record.get("valid")
            rows.append(flow_metrics(flow, truth, valid))
            zero_rows.append(flow_metrics(np.zeros_like(truth), truth, valid))
        return {
            **aggregate_metrics(rows),
            "zero_flow_baseline": aggregate_metrics(zero_rows),
            "num_flow_updates": num_flow_updates,
            "adapted": self.adapted,
            "estimation": (
                f"one pass over {len(records)} pairs; each pair weighs the same; "
                "no resampling, no dispersion estimate"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        weight_decay: float = DEFAULT_WEIGHT_DECAY,
        num_flow_updates: int = DEFAULT_TRAIN_UPDATES,
        gamma: float = DEFAULT_GAMMA,
        seed: int = DEFAULT_SEED,
        freeze_encoders: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning with RAFT's sequence loss, mutating this pipeline.

        Every BatchNorm layer is held in eval mode, as torchvision's reference does in its Sintel stage
        (``--freeze-batch-norm``). With ``freeze_encoders`` the feature and context encoders also keep their
        weights and only the recurrent update block and the upsampling mask predictor train. Gradients are
        clipped to norm 1.0. Frames in a batch must share a size.
        """
        import torch

        validate_dataset(records, epochs=epochs)
        updates = _check_updates(num_flow_updates)
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if (
            isinstance(learning_rate, bool)
            or not isinstance(learning_rate, int | float)
            or not 0.0 < float(learning_rate) <= 1.0
        ):
            raise ValueError(f"learning_rate must be a number in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = not (freeze_encoders and name.startswith(ENCODER_PREFIXES))
        self.frozen_prefixes = ENCODER_PREFIXES if freeze_encoders else ()
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=float(learning_rate), weight_decay=float(weight_decay))

        by_size: dict[tuple[int, int], list[int]] = {}
        for i, record in enumerate(records):
            by_size.setdefault(record["image1"].size, []).append(i)
        epoch_losses: list[float] = []
        for epoch in range(epochs):
            self.model.train()
            _hold_batchnorm(self.model)
            batches = []
            for indices in by_size.values():
                order = [indices[int(j)] for j in rng.permutation(len(indices))]
                batches.extend(order[k : k + batch_size] for k in range(0, len(order), batch_size))
            running, n_batches = 0.0, 0
            for b in rng.permutation(len(batches)):
                batch = [records[i] for i in batches[int(b)]]
                first = [validate_pair(r["image1"], r["image2"])[0] for r in batch]
                second = [validate_pair(r["image1"], r["image2"])[1] for r in batch]
                truth = (
                    torch.from_numpy(np.stack([np.asarray(r["flow"], np.float32) for r in batch]))
                    .permute(0, 3, 1, 2)
                    .to(self.device)
                )
                valid = torch.from_numpy(
                    np.stack(
                        [np.asarray(r.get("valid", np.ones(r["flow"].shape[:2], bool)), bool) for r in batch]
                    )
                ).to(self.device)
                optimizer.zero_grad(set_to_none=True)
                loss = sequence_loss(self._forward(first, second, updates), truth, valid, gamma)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
                optimizer.step()
                running += float(loss.detach().cpu())
                n_batches += 1
            epoch_losses.append(running / max(1, n_batches))
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": epoch_losses[-1]})
        self.model.eval()
        self.adapted = True
        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "weight_decay": float(weight_decay),
            "optimizer": "AdamW",
            "loss": f"RAFT sequence loss (L1, gamma {gamma}, displacements >= {MAX_FLOW:.0f} px ignored)",
            "gradient_clip_norm": 1.0,
            "batchnorm": "frozen (eval mode)",
            "num_flow_updates": updates,
            "seed": seed,
            "precision": "float32",
            "freeze_encoders": freeze_encoders,
            "frozen_prefixes": list(self.frozen_prefixes),
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "device": self.device,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted tensors as one SafeTensors file with the provenance in its metadata.

        Tensors under ``frozen_prefixes`` are left out: they equal the verified base checkpoint, which
        ``load_artifact`` loads first. The artifact is therefore an adapter bound to the base checkpoint.
        """
        from safetensors.torch import save_file

        if not self.adapted:
            raise RuntimeError("nothing to export: the pipeline has not been fine-tuned")
        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        tensors = {
            name: value.detach().cpu().contiguous().clone()
            for name, value in self.model.state_dict().items()
            if not any(name.startswith(prefix) for prefix in self.frozen_prefixes)
        }
        metadata = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "frozen_prefixes": json.dumps(list(self.frozen_prefixes)),
            "base_state_digest": self.base_state_digest or "",
            "notes": notes or "",
        }
        save_file(tensors, str(artifact_path), metadata=metadata)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "tensors": len(tensors),
            "frozen_prefixes": list(self.frozen_prefixes),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def read_artifact_metadata(path: str | Path) -> dict[str, Any]:
        """Read and check the artifact's provenance header without loading any tensor."""
        from safetensors import safe_open

        with safe_open(str(path), framework="pt") as handle:
            metadata = dict(handle.metadata() or {})
        if metadata.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {metadata.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if (metadata.get("model_id"), metadata.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(
                f"artifact was built on {metadata.get('model_id')}@{metadata.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if metadata.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on {metadata.get('model_key')!r}, package pins {MODEL_KEY!r}"
            )
        return {**metadata, "frozen_prefixes": json.loads(metadata["frozen_prefixes"])}

    def apply_artifact(self, path: str | Path) -> None:
        """Load adapter tensors onto this base pipeline; refuse any tensor set that does not fit."""
        from safetensors.torch import load_file

        metadata = self.read_artifact_metadata(path)
        expected_base = metadata.get("base_state_digest") or None
        if expected_base and self.base_state_digest and expected_base != self.base_state_digest:
            raise ValueError("artifact was exported against a different base checkpoint digest")
        tensors = load_file(str(path), device="cpu")
        prefixes = tuple(metadata["frozen_prefixes"])
        result = self.model.load_state_dict(tensors, strict=False)
        if result.unexpected_keys:
            raise ValueError(
                f"artifact carries tensors the model does not have: {result.unexpected_keys[:5]}"
            )
        stray = [key for key in result.missing_keys if not any(key.startswith(p) for p in prefixes)]
        if stray:
            raise ValueError(f"artifact is missing trainable tensors: {stray[:5]}")
        self.model.eval()
        self.adapted = True
        self.frozen_prefixes = prefixes
        self.source = f"artifact:{Path(path).name}"

    @classmethod
    def load_artifact(
        cls, path: str | Path, *, weights_dir: str | Path | None = None, device: str | None = None
    ) -> RaftPipeline:
        """Rebuild an adapted pipeline: verified base checkpoint first, then the adapter tensors."""
        cls.read_artifact_metadata(path)
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir)
        pipe.apply_artifact(path)
        return pipe


def flow_to_rgb(flow: np.ndarray) -> Image.Image:
    """Colour-code a flow field (hue = direction, saturation = speed) with torchvision's flow_to_image."""
    import torch
    from torchvision.utils import flow_to_image

    tensor = torch.from_numpy(np.ascontiguousarray(np.asarray(flow, dtype=np.float32).transpose(2, 0, 1)))
    return Image.fromarray(flow_to_image(tensor).permute(1, 2, 0).numpy())

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the manifest URL on `download.pytorch.org` **and must hash to SHA-256 `unpinned…`**, `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_url_snapshot",
  "formatVersion": 1,
  "modelKey": "raft-large-c-t-skht-v2",
  "modelId": "torchvision/raft_large",
  "revision": "unpinned",
  "files": [
    {
      "path": "raft_large_C_T_SKHT_V2-ff5fadd5.pth",
      "url": "https://download.pytorch.org/models/raft_large_C_T_SKHT_V2-ff5fadd5.pth",
      "bytes": null,
      "sha256": null
    }
  ],
  "totalBytes": null
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained model does on a pair with exact flow

Before adapting anything, inspect the model you start from. The carried `samples` module renders a deterministic frame pair from a textured canvas. The **background** moves by one whole-pixel displacement, and each textured disc or square **object** moves by its own. Because the pair is rendered, the true flow of every pixel is known exactly. A pixel is **valid** when its destination is visible in frame 2; background pixels that an object covers in frame 2, or that leave the frame, are not.

The metrics are the ones optical-flow benchmarks report, computed over valid pixels only:
- **EPE** (end-point error): the mean distance in pixels between the predicted and the true displacement. Lower is better.
- **1px / 3px / 5px**: the fraction of valid pixels whose error is below that many pixels. Higher is better.
- **Angular error**: the mean angle in degrees between the space-time vectors `(u, v, 1)` of the prediction and the truth.

Every score is printed next to the **zero-flow baseline**, which predicts no motion anywhere. Its EPE equals the mean true displacement, so a model is only useful where it beats that row. One pair gives the verdict `sample-sanity`, not a benchmark.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')

NUM_FLOW_UPDATES = 12  # @param {type:"integer"}
DEMO_SEED = 3  # @param {type:"integer"}

print({'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PIXELS': MAX_PIXELS,
       'DEFAULT_FLOW_UPDATES': DEFAULT_FLOW_UPDATES, 'MAX_FLOW': MAX_FLOW})

demo = moving_shapes_pair(DEMO_SEED)
buffer = io.BytesIO()
demo['image1'].save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(demo['image1'].size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'background_shift': demo['background_shift'], 'objects': [(o['kind'], o['shift']) for o in demo['objects']],
       'valid_fraction': round(float(demo['valid'].mean()), 3)})

input_manifest = validate_inputs(demo['image1'], demo['image2'], num_flow_updates=NUM_FLOW_UPDATES)
try:
    validate_inputs(demo['image1'], demo['image2'].resize((128, 128)))
except ValueError as exc:
    input_manifest['findings'].append({'probe': 'frames of different sizes', 'rejected': str(exc)})
print({'verdict': input_manifest['verdict'], 'padding': input_manifest['padding'], 'findings': input_manifest['findings']})

demo_result = pipe.estimate(demo['image1'], demo['image2'], num_flow_updates=NUM_FLOW_UPDATES)
demo_report = evaluation_report(demo_result, demo['flow'], demo['valid'], sample_kind='synthetic')
zero_row = demo_report['baselines'][0]
print(f"{'':<10s} {'EPE':>8s} {'1px':>7s} {'3px':>7s} {'5px':>7s} {'angle':>8s}")
for name, row in (('RAFT', demo_report['metrics']), ('zero-flow', zero_row)):
    print(f"{name:<10s} {row['epe']:>8.3f} {row['1px']:>7.3f} {row['3px']:>7.3f} {row['5px']:>7.3f} {row['angular_error_deg']:>8.2f}")
print({'verdict': demo_report['verdict'], 'valid_pixels': demo_report['metrics']['valid_pixels']})

strip = Image.new('RGB', (4 * 256, 256))
for index, panel in enumerate((demo['image1'], demo['image2'], flow_to_rgb(demo['flow']), flow_to_rgb(demo_result['flow']))):
    strip.paste(panel.resize((256, 256)), (256 * index, 0))
print('frame 1 | frame 2 | true flow | predicted flow (hue = direction, saturation = speed)')
strip

## 5. Recurrent updates and degenerate input probes

RAFT's estimate depends on how many times the update block refines it. torchvision's default is 12 updates, and the builder accepts any count; this pipeline allows 1 to 32. The first table re-scores the demonstration pair at 1, 4, 12 and 32 updates, so you can see where more refinement stops paying for its compute.

Then ask the model about pairs whose answer is known before trusting it:
- **Static pair:** the same textured frame twice. The true flow is zero everywhere, so any predicted motion is error.
- **Blank pair:** two featureless white frames. Motion is undetermined, because nothing can be matched. The cell reports the mean predicted displacement; a large value is motion the model invented.

**What to look for:** EPE close to zero on the static pair, and a small mean displacement on the blank pair.

In [ ]:
iteration_sweep = {}
for updates in (1, 4, 12, 32):
    flow = pipe.estimate(demo['image1'], demo['image2'], num_flow_updates=updates)['flow']
    iteration_sweep[updates] = {k: round(v, 4) for k, v in flow_metrics(flow, demo['flow'], demo['valid']).items() if k != 'valid_pixels'}
    print(f'{updates:>2d} updates', iteration_sweep[updates])

degenerate = {}
for probe in (static_pair(), blank_pair()):
    flow = pipe.estimate(probe['image1'], probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
    speed = np.linalg.norm(flow, axis=2)
    degenerate[probe['id']] = {'epe_vs_zero_truth': round(float(speed.mean()), 4), 'max_displacement_px': round(float(speed.max()), 4)}
print(json.dumps(degenerate, indent=2))

## 6. Labelled adaptation dataset and validation

`flow_dataset` renders a deterministic dataset of `N_PAIRS` pairs like the demonstration pair, each from its own seed, with background displacements up to 6 px and object displacements up to 12 px in each axis. Every record carries `image1`, `image2`, the exact `flow` and the `valid` mask.

`validate_dataset` checks every record before any model runs: the record keys, that both frames share one size inside the ceilings, that the flow has shape `(H, W, 2)` and is finite, and that the valid mask is boolean, has the frame's shape and selects at least one pixel. It returns a dataset manifest with the mean displacement and the mean valid fraction, and a finding for any valid pixel that moves `MAX_FLOW` (400) px or more, which the loss ignores.

In [ ]:
N_PAIRS = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 3  # @param {type:"integer"}

records = flow_dataset(N_PAIRS, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, epochs=EPOCHS)
print(json.dumps(dataset_manifest, indent=2))

preview = Image.new('RGB', (3 * 160, 2 * 160))
for index, record in enumerate(records[:3]):
    preview.paste(record['image1'].resize((160, 160)), (160 * index, 0))
    preview.paste(flow_to_rgb(record['flow']).resize((160, 160)), (160 * index, 160))
preview

## 7. Split and measure the pre-adaptation baseline

The dataset is split at random into a training part (75%, 30 pairs) and a held-out part (25%, 10 pairs). A random split is valid here because every pair is rendered independently. Pairs cut from the same video must be split by video instead, or the held-out frames will share content with the training frames. The held-out pairs are never shown to the optimizer, and no hyperparameter is selected on them.

The fine-tune runs on its own copy of the verified model, `adapter`, so `pipe` stays the pretrained reference. The baseline is that copy's held-out score before any update.

**The pretrained model already beats the zero-flow baseline here.** Textured frames moved by whole pixels are an easy case for RAFT, so a large gap between the two baseline rows is expected. The fine-tune has to improve on the RAFT row, not on zero flow.

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_pairs(records, train_fraction=1.0 - HOLDOUT, seed=SEED)
overlap = {r['id'] for r in train_records} & {r['id'] for r in held_out}
assert not overlap, f'split leaked records: {sorted(overlap)}'
print({'train': len(train_records), 'held_out': len(held_out)})

adapter = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
baseline = adapter.evaluate(held_out, num_flow_updates=NUM_FLOW_UPDATES)
METRIC_KEYS = ('epe', '1px', '3px', '5px', 'angular_error_deg')
print(json.dumps({'pretrained': {k: round(baseline[k], 4) for k in METRIC_KEYS},
                  'zero_flow': {k: round(baseline['zero_flow_baseline'][k], 4) for k in METRIC_KEYS}}, indent=2))

## 8. Bounded fine-tuning

This cell runs the real adaptation step in this runtime. It is gradient fine-tuning with RAFT's **sequence loss**, the loss of the RAFT paper and of torchvision's reference training script. The model returns one flow estimate per recurrent update; the loss is the L1 distance to the true flow over valid pixels for each estimate, weighted by `gamma ** (N - i - 1)` with `gamma = 0.8`, so the last estimate weighs most. Pixels whose true displacement is 400 px or more are ignored, as in the reference.

- **The feature and context encoders are frozen by default.** With `FREEZE_ENCODERS = True` their weights keep the pretrained values, and only the recurrent update block and the upsampling mask predictor train. The cell prints the trainable and total parameter counts.
- **Every BatchNorm layer is held in evaluation mode**, so its running statistics do not drift either, as torchvision's reference does in its Sintel stage (`--freeze-batch-norm`).
- **Schedule:** `EPOCHS` epochs of AdamW at learning rate `2e-5` and weight decay `5e-5`, batch size 2, 12 recurrent updates, gradients clipped to norm 1.0, float32, seed `SEED`.

**Read the loss as optimisation evidence only.** A falling loss says the optimizer is fitting the training pairs; the held-out end-point error in the next section is the task evidence.

In [ ]:
LEARNING_RATE = 2e-5  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
FREEZE_ENCODERS = True  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_encoders=FREEZE_ENCODERS,
    progress=lambda row: print(f"epoch {row['epoch']}/{row['epochs']}  loss {row['loss']:.4f}"),
)
print(json.dumps({key: run[key] for key in ('freeze_encoders', 'batchnorm', 'trainable_parameters', 'total_parameters', 'epochs',
                                         'batch_size', 'learning_rate', 'optimizer', 'loss', 'precision', 'device')}, indent=2))

## 9. Evaluate on the held-out split

`evaluate` re-runs on the same held-out pairs with the same number of updates as the baseline, so the rows are comparable. Each metric is the mean over pairs, each pair weighing the same; `pixel_weighted_epe` weighs every valid pixel the same instead. These are tutorial metrics from one pass over 10 synthetic pairs, with no dispersion estimate. A change of a few hundredths of a pixel is within what a different seed can move.

In [ ]:
adapted = adapter.evaluate(held_out, num_flow_updates=NUM_FLOW_UPDATES)
print(f"{'metric':<18s} {'zero-flow':>10s} {'pretrained':>11s} {'adapted':>10s} {'change':>10s}")
for key in METRIC_KEYS + ('pixel_weighted_epe',):
    zero = baseline['zero_flow_baseline'][key]
    print(f"{key:<18s} {zero:>10.4f} {baseline[key]:>11.4f} {adapted[key]:>10.4f} {adapted[key] - baseline[key]:>+10.4f}")

## 10. Inference on unseen pairs

Three new pairs come from a seed the dataset never used (`NEW_DATA_SEED = 99`). The pretrained and the adapted pipelines estimate flow on each, and both are scored against the exact flow.

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = flow_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for record in new_records:
    row = {'id': record['id'], 'mean_true_displacement_px': round(float(np.linalg.norm(record['flow'], axis=2)[record['valid']].mean()), 3)}
    for name, model in (('pretrained', pipe), ('adapted', adapter)):
        flow = model.estimate(record['image1'], record['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
        row[f'{name}_epe'] = round(flow_metrics(flow, record['flow'], record['valid'])['epe'], 4)
    new_data_rows.append(row)
    print(json.dumps(row))

## 11. Adapter export, fresh reload, and equivalence check

`save_artifact` writes `outputs/raft_adapter.safetensors`: every tensor the fine-tune could change, plus a metadata header naming the base model, its pinned checkpoint SHA-256 and the frozen prefixes. With the encoders frozen, their tensors are left out because they equal the verified base checkpoint.

`load_artifact` then builds a **fresh** pipeline from the verified base checkpoint, loads the adapter tensors onto it, and refuses an adapter whose format, base identity, base digest or tensor set does not fit. The cell compares the reloaded flow with the in-memory model's on an unseen pair, with a stated tolerance: loading succeeding is not the check, reproducing the flow is.

In [ ]:
artifact_path = OUTPUTS / 'raft_adapter.safetensors'
descriptor = adapter.save_artifact(artifact_path, notes='RAFT-Large moving-shapes adaptation tutorial adapter')
print(json.dumps(descriptor, indent=2))

reloaded = RaftPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'frozen_prefixes': list(reloaded.frozen_prefixes)})

TOLERANCE_PX = 1e-3
probe = new_records[0]
flow_orig = adapter.estimate(probe['image1'], probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
flow_reloaded = reloaded.estimate(probe['image1'], probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
difference = float(np.linalg.norm(flow_orig - flow_reloaded, axis=2).mean())
assert difference <= TOLERANCE_PX, f'reloaded flow differs by {difference:.6f} px on average'
reload_check = {'pair': probe['id'], 'mean_endpoint_difference_px': difference, 'tolerance_px': TOLERANCE_PX, 'equivalent': True}
print(reload_check)

## 12. Write machine-readable outputs and provenance

The cell writes:
- `outputs/raft_optical_flow_input_manifest.json`
- `outputs/raft_optical_flow_evaluation_report.json`
- `outputs/raft_optical_flow_result.json` (identity, runtime versions, device, iteration sweep, probes, dataset manifest, split, baseline and adapted metrics, fine-tuning configuration, new-data rows, adapter descriptor and reload check)
- `outputs/raft_optical_flow_metrics.csv` (one row per held-out and unseen pair, pretrained and adapted EPE)
- `outputs/raft_optical_flow_flow.flo` and `outputs/raft_optical_flow_flow.npy` (the demonstration pair's predicted flow)
- `outputs/raft_optical_flow_flow.png` (the same flow, colour-coded)
- `outputs/raft_adapter.safetensors` (written in Section 11)

In [ ]:
import csv

with open(OUTPUTS / 'raft_optical_flow_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'raft_optical_flow_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(demo_report, f, indent=2)

write_flo(OUTPUTS / 'raft_optical_flow_flow.flo', demo_result['flow'])
np.save(OUTPUTS / 'raft_optical_flow_flow.npy', demo_result['flow'])
flow_to_rgb(demo_result['flow']).save(OUTPUTS / 'raft_optical_flow_flow.png')
assert np.array_equal(read_flo(OUTPUTS / 'raft_optical_flow_flow.flo'), demo_result['flow'])

with open(OUTPUTS / 'raft_optical_flow_metrics.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['split', 'pair', 'pretrained_epe', 'adapted_epe', 'zero_flow_epe'])
    for split, group in (('held_out', held_out), ('unseen', new_records)):
        for record in group:
            scores = [flow_metrics(m.estimate(record['image1'], record['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow'], record['flow'], record['valid'])['epe'] for m in (pipe, adapter)]
            zero = flow_metrics(np.zeros_like(record['flow']), record['flow'], record['valid'])['epe']
            writer.writerow([split, record['id'], *(f'{v:.4f}' for v in (*scores, zero))])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'weights_url': WEIGHTS_URL,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                'cuda': torch.cuda.is_available()},
    'device': pipe.device,
    'num_flow_updates': NUM_FLOW_UPDATES,
    'demo_metrics': demo_report['metrics'],
    'iteration_sweep': iteration_sweep,
    'degenerate_probes': degenerate,
    'adaptation': {
        'dataset': dataset_manifest,
        'dataset_seed': DATASET_SEED,
        'split': {'train': len(train_records), 'held_out': len(held_out), 'seed': SEED, 'holdout': HOLDOUT},
        'finetune': run,
        'baseline': baseline,
        'adapted': adapted,
        'new_data': new_data_rows,
        'artifact': descriptor,
        'reload_check': reload_check,
    },
}
with open(OUTPUTS / 'raft_optical_flow_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

for p in sorted(OUTPUTS.iterdir()):
    if p.is_file():
        print(f'  {p.name:<44s} {p.stat().st_size:>12,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Both branches are off by default, so `Run all` never stops here. Before you turn one on, read the contract:

- **Frame-pair branch** (`USE_BYOD_IMAGE`): two image files PIL can open, of the same size, each side between `MIN_IMAGE_SIDE` (128) and `MAX_IMAGE_SIDE` (1024) px and at most `MAX_PIXELS` (786,432) pixels. Frame 1 comes before frame 2 in time. The pair runs through `validate_inputs`, `estimate` and `evaluation_report`; the report is `not-measurable`, because no ground-truth flow comes with it.
- **Dataset branch** (`USE_BYOD_DATASET`): a directory with `pairs.json` (a list of `{'frame1', 'frame2', 'flow', 'valid'}` objects) and the files it names, at least 2 records and at most 2,000. `flow` is a Middlebury `.flo` file or a `.npy` array of shape `(H, W, 2)`; `valid` is an optional single-channel PNG whose non-zero pixels are valid. File names must stay inside the directory. The branch runs the same validate → split → baseline → fine-tune → evaluate → export → reload stages as the sample.

Set `BYOD_IMAGE1_PATH` and `BYOD_IMAGE2_PATH`, or `BYOD_DATASET_DIR`, to read from a mounted or local location; leave them empty on Colab to get an upload dialog instead (for the pair branch, upload both frames at once; they are ordered by file name). Uploaded files are written under `outputs/byod/` in this runtime and are not sent anywhere else. The first lines of the cell show the validator refusing two malformed inputs with messages that name the failed rule.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE1_PATH = ""  # @param {type:"string"}
BYOD_IMAGE2_PATH = ""  # @param {type:"string"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_DIR = ""  # @param {type:"string"}

_probe = static_pair()
for desc, probe_call in (
    ('non-image object', lambda: validate_inputs('/not/an/image.png', _probe['image2'])),
    ('flow of the wrong shape', lambda: validate_dataset([{'image1': _probe['image1'], 'image2': _probe['image2'],
                                                          'flow': np.zeros((8, 8, 2), np.float32)}])),
):
    try:
        probe_call()
    except (TypeError, ValueError) as exc:
        print(f'refused as expected: {desc} -> {type(exc).__name__}: {exc}')

BYOD_DIR = OUTPUTS / 'byod'

def _upload_into(target):
    from google.colab import files  # type: ignore[import-not-found]
    target.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (target / Path(name).name).write_bytes(data)
    return target

if USE_BYOD_IMAGE:
    if BYOD_IMAGE1_PATH and BYOD_IMAGE2_PATH:
        frame_paths = [Path(BYOD_IMAGE1_PATH), Path(BYOD_IMAGE2_PATH)]
    else:
        frame_paths = sorted(p for p in _upload_into(BYOD_DIR / 'pair').iterdir() if p.is_file())[:2]
    if len(frame_paths) != 2:
        raise ValueError(f'the frame-pair branch needs exactly two frames, got {len(frame_paths)}')
    byod_frames = []
    for frame_path in frame_paths:
        with Image.open(frame_path) as handle:
            byod_frames.append(handle.convert('RGB'))
    print(validate_inputs(*byod_frames, num_flow_updates=NUM_FLOW_UPDATES)['verdict'], [p.name for p in frame_paths])
    byod_result = pipe.estimate(*byod_frames, num_flow_updates=NUM_FLOW_UPDATES)
    byod_report = evaluation_report(byod_result, sample_kind='byod')
    print({'verdict': byod_report['verdict'], 'mean_predicted_displacement_px': round(byod_report['mean_predicted_displacement_px'], 3)})
    write_flo(OUTPUTS / 'byod_flow.flo', byod_result['flow'])
    flow_to_rgb(byod_result['flow']).save(OUTPUTS / 'byod_flow.png')
else:
    print('BYOD frame-pair branch is off; set USE_BYOD_IMAGE = True to estimate flow between your own two frames.')

if USE_BYOD_DATASET:
    dataset_dir = Path(BYOD_DATASET_DIR) if BYOD_DATASET_DIR else _upload_into(BYOD_DIR / 'dataset')
    byod_records = read_flow_records(dataset_dir)
    byod_manifest = validate_dataset(byod_records, epochs=EPOCHS)
    print(json.dumps(byod_manifest, indent=2))
    byod_train, byod_held = split_pairs(byod_records, train_fraction=1.0 - HOLDOUT, seed=SEED)
    byod_pipe = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
    byod_baseline = byod_pipe.evaluate(byod_held, num_flow_updates=NUM_FLOW_UPDATES)
    byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_encoders=FREEZE_ENCODERS)
    byod_adapted = byod_pipe.evaluate(byod_held, num_flow_updates=NUM_FLOW_UPDATES)
    print({key: (round(byod_baseline['zero_flow_baseline'][key], 4), round(byod_baseline[key], 4), round(byod_adapted[key], 4)) for key in ('epe', '3px')})
    byod_descriptor = byod_pipe.save_artifact(OUTPUTS / 'byod_raft_adapter.safetensors', notes='BYOD adaptation adapter')
    RaftPipeline.load_artifact(OUTPUTS / 'byod_raft_adapter.safetensors', weights_dir=WEIGHTS_DIR)
    print('BYOD adapter exported and reloaded:', byod_descriptor['sha256'][:16])
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt RAFT on your own frame pairs with ground-truth flow.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned torchvision checkpoint for `torchvision/raft_large` was verified against a committed SHA-256 manifest before loading. The pretrained model estimated flow on a rendered pair with exact ground truth, was re-scored at four update counts, and was probed with a static and a featureless pair. A labelled dataset of rendered pairs was then used to fine-tune the recurrent update block with RAFT's sequence loss, with the encoders and every BatchNorm layer frozen, and the held-out split was scored by end-point error before and after, next to the zero-flow baseline. The adapter was exported, reloaded onto a fresh base model and checked against the in-memory flow.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the rendered pairs it was fitted to. The held-out EPE is measured on 10 rendered pairs and carries no dispersion estimate.

**Reproducibility.** Seeds are form fields (`DEMO_SEED`, `DATASET_SEED`, `SEED`, `NEW_DATA_SEED`), and the run is float32 with no data augmentation. GPU kernels (the correlation lookup's `grid_sample` backward pass in particular) are not forced to be deterministic, so repeated GPU runs can differ in the last digits of the loss and the scores.

**Try next.** Change `FREEZE_ENCODERS` to `False` and compare held-out EPE and runtime, or change `NUM_FLOW_UPDATES` and watch the EPE and the time per pair. To transfer the workflow, point `BYOD_DATASET_DIR` at frame pairs with ground-truth flow from your own domain, for example rendered from a simulator.

## References

- Teed, Z. and Deng, J. (2020). *RAFT: Recurrent All-Pairs Field Transforms for Optical Flow.* ECCV 2020. [arXiv:2003.12039](https://arxiv.org/abs/2003.12039).
- torchvision model documentation: [raft_large](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.optical_flow.raft_large.html); upstream repository [pytorch/vision](https://github.com/pytorch/vision) — BSD-3-Clause; training recipe [references/optical_flow](https://github.com/pytorch/vision/tree/main/references/optical_flow).
- Butler, D. J., Wulff, J., Stanley, G. B. and Black, M. J. (2012). *A Naturalistic Open Source Movie for Optical Flow Evaluation.* ECCV 2012 — the Sintel benchmark.
- Baker, S., Scharstein, D., Lewis, J. P., Roth, S., Black, M. J. and Szeliski, R. (2011). *A Database and Evaluation Methodology for Optical Flow.* IJCV 92 — the Middlebury `.flo` format and the angular error.
- Repository model card: https://github.com/kurtvalcorza/raft-optical-flow-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/raft-optical-flow-pipeline`](https://github.com/kurtvalcorza/raft-optical-flow-pipeline) — source repository for this pipeline.